# Suitability Score Documentation

This notebook documents how the cafe suitability score is calculated in this project, how it is shown in the frontend, what each formula means, and which reference articles the ideas were derived from.

Important honesty note:
- Some formulas in this project are direct engineering heuristics created for this student project.
- Those heuristics were inspired by ideas from retail site-selection, GIS, accessibility, competition, and multi-criteria decision-making literature.
- They are **not** copied verbatim from one single paper.
- Wherever that is the case, this notebook explicitly says `project heuristic inspired by literature`.

## 1. End-to-End Flow

The score shown in the frontend follows this path:

1. User clicks a location on the map.
2. Backend computes location features such as:
   - population density
   - road accessibility
   - bus-stop / school / hospital counts
   - foot-traffic proxy
   - competition pressure
3. Backend converts these into the 8 model input features.
4. Backend sends the features into Random Forest and XGBoost models.
5. Backend averages the two model predictions into one ensemble score.
6. Backend returns that score in the API response.
7. Frontend displays the returned score, label, and colored score circle.

Relevant project files:
- `backend/api/views.py`
- `backend/ml_engine/suitability_predictor.py`
- `frontend/js/map.js`
- `frontend/map.html`

## 2. Heuristic Suitability Score in the API

A simple heuristic score is first computed in the backend. In the current app this is **not normally the final displayed score**; it mainly acts as a backup / fallback value when needed.

Source: `backend/api/views.py`

### Formula

```text
weighted_competitors = total_competitors + 1.5 * same_type_competitors

competitor_score = max(0, 1 - weighted_competitors / 20) * 4
road_score       = min(1, road_m / 3000) * 3
pop_score        = min(1, pop_density / 15000) * 3

heuristic_suitability_score = round(competitor_score + road_score + pop_score, 2)
```

### How each part works

- `weighted_competitors`
  Same-type competitors are penalized more heavily than other cafes because they compete more directly for the same customers.

- `competitor_score`
  This converts competition into a score from `0` to `4`.
  If competition is low, the score is high.
  If competition becomes large enough, the score bottoms out at `0`.

- `road_score`
  This gives up to `3` points based on estimated road presence / access.
  More road availability is treated as better accessibility.

- `pop_score`
  This gives up to `3` points based on surrounding population density.
  Denser areas are assumed to imply stronger demand potential.

### Derivation note

This exact weighted formula is a **project heuristic inspired by literature**, not a published formula copied directly from one article.

Literature support for the chosen ideas:
- demand / population matters for retail success
- accessibility matters for retail success
- competition matters for retail success

Reference articles:
- Roig-Tierno, N., Baviera-Puig, A., Buitrago-Vera, J., and Mas-Verdu, F. (2013). *The retail site location decision process using GIS and the analytical hierarchy process*. Applied Geography, 40, 191-198. DOI: https://doi.org/10.1016/j.apgeog.2013.03.005
- Wibisono, Y. Y. and Marella, S. (2020). *A decision making model for selection of cafe location: An ANP approach*. Journal of Physics: Conference Series, 1477, 052030. DOI: https://doi.org/10.1088/1742-6596/1477/5/052030

## 3. Engineered Features Used for the ML Score

The backend computes several intermediate formulas before sending features to the ML models.

Source: `backend/api/views.py`

### 3.1 Nearest-road feature

```text
nearest_road_for_features = nearest_main_road_m
if nearest_main_road_m is missing:
    nearest_road_for_features = clamp(road_m / 2, 50, 3000)
```

How it works:
- Uses actual nearest main-road distance if available.
- Otherwise falls back to a bounded estimate.

Derivation:
- Project heuristic for robustness.
- Concept inspired by accessibility-based retail location work.

### 3.2 Accessibility score

```text
road_access_score = clamp(10 - nearest_road_for_features / 150, 0, 10)
bus_access_bonus  = min(2.5, bus_stops_within_500m * 0.35)
school_bonus      = min(1.5, schools_within_500m * 0.15)
hospital_bonus    = min(1.0, hospitals_within_500m * 0.15)

accessibility_score = clamp(
    road_access_score + bus_access_bonus + school_bonus + hospital_bonus,
    0,
    10
)
```

How it works:
- Closer to a main road means a higher `road_access_score`.
- Nearby bus stops increase public accessibility.
- Nearby schools and hospitals act as destination generators and increase surrounding activity.
- The total is capped at `10` so the feature stays in a stable range.

Derivation:
- Exact weights are a project heuristic.
- The idea that accessibility and nearby activity centers affect store suitability is supported by retail location literature.

References:
- Wang, L., Fan, H., and Wang, Y. (2018). *Site Selection of Retail Shops Based on Spatial Accessibility and Hybrid BP Neural Network*. ISPRS International Journal of Geo-Information, 7(6), 202. DOI: https://doi.org/10.3390/ijgi7060202
- Wibisono, Y. Y. and Marella, S. (2020). *A decision making model for selection of cafe location: An ANP approach*. DOI: https://doi.org/10.1088/1742-6596/1477/5/052030

### 3.3 Foot-traffic score

```text
density_signal       = min(4.0, pop_density / 5000)
transit_signal       = min(3.0, bus_stops_within_500m * 0.4)
institutional_signal = min(2.0, schools_within_500m * 0.2)
commerce_signal      = min(2.0, total_competitors * 0.12)
road_signal          = min(2.0, max(0.0, 2.0 - nearest_road_for_features / 300))

foot_traffic_score = clamp(
    density_signal + transit_signal + institutional_signal + commerce_signal + road_signal,
    0,
    10
)
```

How it works:
- `density_signal`: denser population implies more potential passers-by.
- `transit_signal`: more bus stops imply more movement and access.
- `institutional_signal`: schools may generate concentrated nearby activity.
- `commerce_signal`: existing cafes can indicate active commercial zones.
- `road_signal`: closer road access improves practical movement.

Important interpretation:
- The app does not directly measure real pedestrian counts.
- It builds a **proxy** for foot traffic using surrounding urban signals.

Derivation:
- Project heuristic proxy inspired by retail and cafe site-selection criteria such as traffic density, market size, accessibility, and closeness to market.

References:
- Wibisono, Y. Y. and Marella, S. (2020). *A decision making model for selection of cafe location: An ANP approach*. DOI: https://doi.org/10.1088/1742-6596/1477/5/052030
- Roig-Tierno, N., Baviera-Puig, A., Buitrago-Vera, J., and Mas-Verdu, F. (2013). *The retail site location decision process using GIS and the analytical hierarchy process*. DOI: https://doi.org/10.1016/j.apgeog.2013.03.005

### 3.4 Competition pressure

```text
competition_pressure = clamp(
    total_competitors * 0.30 +
    same_type_competitors * 0.85 +
    min(2.0, avg_rating * 0.25) +
    min(2.5, same_type_avg_rating * 0.55),
    0,
    10
)
```

How it works:
- More nearby cafes increase pressure.
- Same-type cafes count more because they are closer substitutes.
- Higher ratings increase pressure because strong incumbents are harder to beat.
- The total is capped to keep the feature bounded.

Derivation:
- Project heuristic inspired by geocompetition and business rivalry concepts in retail location studies.

References:
- Roig-Tierno, N., Baviera-Puig, A., Buitrago-Vera, J., and Mas-Verdu, F. (2013). *The retail site location decision process using GIS and the analytical hierarchy process*. DOI: https://doi.org/10.1016/j.apgeog.2013.03.005
- Wang, L., Fan, H., and Wang, Y. (2018). *Site Selection of Retail Shops Based on Spatial Accessibility and Hybrid BP Neural Network*. DOI: https://doi.org/10.3390/ijgi7060202

### 3.5 Amenity density

```text
amenity_count = schools_within_500m + hospitals_within_500m + bus_stops_within_500m
area_km2 = pi * (0.5 ^ 2)
osm_amenity_density_500m = amenity_count / area_km2
```

How it works:
- Counts selected amenities inside a 500 m radius.
- Divides by circular area to get a simple density-per-km^2 measure.

Derivation:
- Project engineering formula using standard area-of-circle geometry.
- The idea of using surrounding POI / amenity concentration as a location signal is literature-inspired.

Reference:
- Wang, L., Fan, H., and Wang, Y. (2018). *Site Selection of Retail Shops Based on Spatial Accessibility and Hybrid BP Neural Network*. DOI: https://doi.org/10.3390/ijgi7060202

## 4. Final Model Input Features

Before prediction, the app builds the 8 regression features below.

Source: `backend/api/views.py`, function `_build_regression_features(...)`

### Formula

```text
population_density       = clamp(pop_density / 1000, 0, 10)
accessibility_score      = clamp(accessibility_score, 0, 10)
foot_traffic_score       = clamp(foot_traffic_score, 0, 10)
competition_effective    = clamp(10 - competition_pressure, 0, 10)
bus_stops_within_500m    = clamp(bus_stops_within_500m, 0, 10)
osm_amenity_density_500m = clamp(osm_amenity_density_500m, 0, 10)
nearby_schools           = clamp(schools_within_500m, 0, 10)
nearby_hospitals         = clamp(hospitals_within_500m, 0, 10)
```

How it works:
- The model expects bounded features.
- `population_density / 1000` compresses census density into a smaller model-friendly range.
- `competition_effective = 10 - competition_pressure` flips the direction so lower competition pressure becomes a stronger positive factor.

Derivation:
- This is a preprocessing / normalization design choice for the trained model.
- It is partly inspired by multi-criteria scoring practice, where criteria are first normalized and then combined or learned.

Reference:
- Kasperczyk, N. and Knickel, K. (2022/2026 guide PDF citing Saaty). *The Analytic Hierarchy Process (AHP)*. IIED guide PDF: https://www.iied.org/sites/default/files/pdfs/2022-02/20781G.pdf
- Harker, P. T. and Vargas, L. G. (1987). *The Theory of Ratio Scale Estimation: Saaty's Analytic Hierarchy Process*. Management Science, 33(11), 1383-1403. DOI: https://doi.org/10.1287/mnsc.33.11.1383

## 5. ML Prediction Formula

Source: `backend/ml_engine/suitability_predictor.py`

### Model features

The trained models use these 8 inputs:

```text
[
  population_density,
  accessibility_score,
  foot_traffic_score,
  competition_effective,
  bus_stops_within_500m,
  osm_amenity_density_500m,
  nearby_schools,
  nearby_hospitals
]
```

### Prediction formula

```text
features_scaled = scaler.transform(features_array)

rf_score  = RandomForest.predict(features_scaled)
xgb_score = XGBoost.predict(features_scaled)

ensemble_score = clip((rf_score + xgb_score) / 2, 0, 10)
```

How it works:
- `scaler.transform(...)` applies the same scaling used during training.
- Random Forest predicts one score.
- XGBoost predicts another score.
- The app averages the two to reduce single-model bias and improve stability.
- The final result is clipped to stay in the `0` to `10` suitability range.

Why averaging is reasonable:
- Ensemble learning combines multiple learners to improve generalization and robustness.
- In this project, the ensemble is a simple mean of two tree-based regressors.

Reference articles:
- Breiman, L. (2001). *Random Forests*. Machine Learning, 45, 5-32. DOI: https://doi.org/10.1023/A:1010933404324
- Friedman, J. H. (2001). *Greedy Function Approximation: A Gradient Boosting Machine*. Annals of Statistics, 29(5), 1189-1232. DOI: https://doi.org/10.1214/aos/1013203451
- Chen, T. and Guestrin, C. (2016). *XGBoost: A Scalable Tree Boosting System*. Proceedings of the 22nd ACM SIGKDD International Conference on Knowledge Discovery and Data Mining, 785-794. DOI: https://doi.org/10.1145/2939672.2939785
- Natras, R., Soja, B., and Schmidt, M. (2022). *Ensemble Machine Learning of Random Forest, AdaBoost and XGBoost for Vertical Total Electron Content Forecasting*. Remote Sensing, 14(15), 3547. DOI: https://doi.org/10.3390/rs14153547

### Confidence formula

```text
confidence = clamp(1 - abs(rf_score - xgb_score) / 10, 0, 1)
```

How it works:
- If RF and XGB agree closely, confidence is higher.
- If they disagree strongly, confidence is lower.

Derivation:
- Project heuristic confidence measure.
- Not taken directly from a single reference article.

## 6. Fallback Score Formula

If the trained models or scaler are unavailable, the backend uses a manual weighted formula.

Source: `backend/ml_engine/suitability_predictor.py`

### Formula

```text
population_component    = min(10, population_density) * 0.25
accessibility_component = min(10, accessibility_score) * 0.20
foot_traffic_component  = min(10, foot_traffic_score) * 0.20
competition_penalty     = min(10, competition_effective) * 0.15
bus_component           = min(10, bus_stops_within_500m * 0.1) * 0.10
amenity_component       = min(10, osm_amenity_density_500m * 0.01) * 0.05
schools_component       = min(10, nearby_schools * 0.1) * 0.03
hospitals_component     = min(10, nearby_hospitals * 0.1) * 0.02

fallback_score = clamp(
    population_component +
    accessibility_component +
    foot_traffic_component +
    bus_component +
    amenity_component +
    schools_component +
    hospitals_component -
    competition_penalty,
    0,
    10
)
```

How it works:
- This is a weighted additive model.
- Positive demand and accessibility signals add score.
- Competition reduces score.
- All terms are bounded so the result remains stable.

Derivation:
- Project heuristic weighted scoring model.
- Conceptually consistent with weighted multi-criteria decision analysis and AHP-style aggregation.

References:
- Roig-Tierno, N., Baviera-Puig, A., Buitrago-Vera, J., and Mas-Verdu, F. (2013). *The retail site location decision process using GIS and the analytical hierarchy process*. DOI: https://doi.org/10.1016/j.apgeog.2013.03.005
- Harker, P. T. and Vargas, L. G. (1987). *The Theory of Ratio Scale Estimation: Saaty's Analytic Hierarchy Process*. DOI: https://doi.org/10.1287/mnsc.33.11.1383

## 7. Suitability Level Formula

Source: `backend/ml_engine/suitability_predictor.py`

```text
if score >= 7:
    level = 'High Suitability'
elif score >= 4:
    level = 'Medium Suitability'
else:
    level = 'Low Suitability'
```

How it works:
- `7-10`: strong site
- `4-6.99`: moderate site
- `<4`: weak site

Derivation:
- Project classification threshold design.
- Not directly taken from a journal article.

## 8. Distance Formula Used in Nearby Calculations

The project uses the haversine distance to compute distances between latitude-longitude points.

Source: `backend/api/views.py`, function `haversine_distance(...)`

### Formula

```text
dlon = lon2 - lon1
dlat = lat2 - lat1

a = sin(dlat/2)^2 + cos(lat1) * cos(lat2) * sin(dlon/2)^2
c = 2 * asin(sqrt(a))
distance = R * c
```

How it works:
- Computes great-circle distance on the Earth surface.
- Appropriate for coordinate-based proximity checks like nearby cafes and amenities.

Reference article:
- Muttaqin, A., Murtopo, A. A., Syefudin, and Gunawan. (2024). *Application of the haversine formula method to determine the closest distance to a minimarket*. Jurnal Mandiri IT. DOI: https://doi.org/10.35335/mandiri.v13i1.293

## 9. How the Frontend Shows the Score

The frontend does not compute the main suitability score itself. It displays the value returned by the backend.

Sources:
- `frontend/map.html`
- `frontend/js/map.js`

### Displayed values

The frontend reads:

```text
main displayed score = data.suitability.score
displayed label      = data.suitability.level
RF model score       = data.prediction.random_forest_score
XGB model score      = data.prediction.xgboost_score
ensemble score       = data.prediction.ensemble_score
```

### Frontend formatting formula

```text
scoreEl.textContent = numericScore.toFixed(2)
```

How it works:
- Always shows the main score to 2 decimal places.
- The visible main score is the backend's final suitability score.

### Score circle formula

```text
normalizedScore = clamp(score, 0, 10)
fillPercent     = normalizedScore * 10
```

Color rule:
- if `score < 4`: red
- if `4 <= score < 7`: yellow
- if `score >= 7`: green

Rendered as:

```text
conic-gradient(color 0% fillPercent%, gray fillPercent% 100%)
```

How it works:
- A score of `8.3` fills about `83%` of the circle.
- The circle is only a visualization layer.
- It does not affect the numeric score.

## 10. Formula-to-Reference Mapping Summary

| Formula / Block | Type | How it was derived |
| --- | --- | --- |
| Heuristic suitability score | Project heuristic | Inspired by retail location literature using demand, accessibility, and competition |
| Accessibility score | Project heuristic | Inspired by accessibility-based retail site-selection studies |
| Foot-traffic proxy score | Project heuristic | Inspired by traffic density, market size, access, and closeness-to-market criteria |
| Competition pressure | Project heuristic | Inspired by geocompetition and direct-rival effects |
| Amenity density | Engineering formula | Standard density = count / area |
| Feature normalization | Engineering / preprocessing | Needed to align live features with model training scale |
| RF and XGB predictions | ML model formula | Based on Random Forest and gradient boosting / XGBoost literature |
| Ensemble mean | Project ensemble rule | Simple averaging inspired by ensemble-learning practice |
| Confidence | Project heuristic | Based on model agreement |
| Haversine distance | Standard geographic formula | Derived from haversine / great-circle distance literature |
| Frontend circle fill | UI formula | Pure display logic for visualization |

This is the most accurate way to describe the project academically:

> The app uses a literature-inspired scoring framework, but several exact weights and caps are project-specific heuristic design choices rather than formulas directly copied from a single paper.

## 11. Reference List

1. Roig-Tierno, N., Baviera-Puig, A., Buitrago-Vera, J., and Mas-Verdu, F. (2013). *The retail site location decision process using GIS and the analytical hierarchy process*. Applied Geography, 40, 191-198. DOI: https://doi.org/10.1016/j.apgeog.2013.03.005

2. Wibisono, Y. Y. and Marella, S. (2020). *A decision making model for selection of cafe location: An ANP approach*. Journal of Physics: Conference Series, 1477, 052030. DOI: https://doi.org/10.1088/1742-6596/1477/5/052030

3. Wang, L., Fan, H., and Wang, Y. (2018). *Site Selection of Retail Shops Based on Spatial Accessibility and Hybrid BP Neural Network*. ISPRS International Journal of Geo-Information, 7(6), 202. DOI: https://doi.org/10.3390/ijgi7060202

4. Harker, P. T. and Vargas, L. G. (1987). *The Theory of Ratio Scale Estimation: Saaty's Analytic Hierarchy Process*. Management Science, 33(11), 1383-1403. DOI: https://doi.org/10.1287/mnsc.33.11.1383

5. Kasperczyk, N. and Knickel, K. *The Analytic Hierarchy Process (AHP)*. IIED guide PDF. URL: https://www.iied.org/sites/default/files/pdfs/2022-02/20781G.pdf

6. Breiman, L. (2001). *Random Forests*. Machine Learning, 45, 5-32. DOI: https://doi.org/10.1023/A:1010933404324

7. Friedman, J. H. (2001). *Greedy Function Approximation: A Gradient Boosting Machine*. Annals of Statistics, 29(5), 1189-1232. DOI: https://doi.org/10.1214/aos/1013203451

8. Chen, T. and Guestrin, C. (2016). *XGBoost: A Scalable Tree Boosting System*. Proceedings of the 22nd ACM SIGKDD International Conference on Knowledge Discovery and Data Mining, 785-794. DOI: https://doi.org/10.1145/2939672.2939785

9. Natras, R., Soja, B., and Schmidt, M. (2022). *Ensemble Machine Learning of Random Forest, AdaBoost and XGBoost for Vertical Total Electron Content Forecasting*. Remote Sensing, 14(15), 3547. DOI: https://doi.org/10.3390/rs14153547

10. Muttaqin, A., Murtopo, A. A., Syefudin, and Gunawan. (2024). *Application of the haversine formula method to determine the closest distance to a minimarket*. Jurnal Mandiri IT. DOI: https://doi.org/10.35335/mandiri.v13i1.293